In [1]:
%%file producer2.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer2.py


In [ ]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    print(message)
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź amount > 1000, jeśli tak — wypisz ALERT

In [1]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
from collections import defaultdict, deque
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='velocity-detector-v1',
    auto_offset_reset='earliest'
)

# Dla każdego user_id trzymamy kolejkę timestampów z ostatnich 60 sekund
user_events = defaultdict(deque)

WINDOW_SECONDS = 60
THRESHOLD = 3

print("Nasłuchuję anomalii prędkości: > 3 transakcje / 60 s dla tego samego user_id...")

for message in consumer:
    event = message.value

    user_id = event.get("user_id")
    tx_id = event.get("tx_id")
    amount = event.get("amount")
    store = event.get("store")
    category = event.get("category")
    ts_raw = event.get("timestamp")

    if not user_id or not ts_raw:
        print(f"Pominięto niepoprawne zdarzenie: {event}")
        continue

    try:
        # Producent wg laba wysyła aktualny czas ISO
        event_time = datetime.fromisoformat(ts_raw)
    except ValueError:
        print(f"Niepoprawny timestamp w zdarzeniu: {event}")
        continue

    queue = user_events[user_id]
    queue.append(event_time)

    # Usuwamy wszystko starsze niż 60 sekund względem bieżącego zdarzenia
    window_start = event_time - timedelta(seconds=WINDOW_SECONDS)
    while queue and queue[0] < window_start:
        queue.popleft()

    tx_count_in_window = len(queue)

    if tx_count_in_window > THRESHOLD:
        print(
            f"ALERT: user={user_id} wykonał {tx_count_in_window} transakcji "
            f"w ciągu ostatnich {WINDOW_SECONDS} s | "
            f"tx_id={tx_id} | amount={amount} PLN | {store} | {category} | ts={ts_raw}"
        )
    else:
        print(
            f"OK: user={user_id} | liczba transakcji w oknie 60 s = {tx_count_in_window}"
        )

Writing consumer_velocity.py
